# GastroNet — NB2: ViT-only baseline (ViT-Small)

**Prerequisite: NB0 must already have been run on this account**, including
the duplicate-check/v2-split cells, so `dataset_split_v2.json` exists on
Drive. This notebook does not generate any split or define checkpoint
functions — it only imports and uses `checkpoint_utils.py`.

This notebook trains the ViT-only branch (`model_family = "vit_only_v2"`) as
one of the ablation baselines, against the corrected `dataset_split_v2.json`
(the duplicate-file leak found in v1 is fixed in v2 — see the project
handoff doc, Section 4). Trains 3 seeds (42, 123, 7) with early stopping,
since single-seed results are not treated as reliable given how tightly all
candidate models cluster on this dataset (handoff doc, Section 1/3/9).


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os, sys, json

# These must be IDENTICAL to what you set in NB0 on this account.
ACCOUNT_TAG      = "acct_A"
EXPERIMENTS_ROOT = "/content/drive/MyDrive/gastronet_experiments"
NOTEBOOK_NAME    = "NB2_vit_only_baseline"
RAW_DATASET_DIR  = "/content/drive/MyDrive/gastronet_raw_dataset"

# This notebook's own identity within the experiment structure.
# "_v2" suffix marks results as trained against dataset_split_v2.json,
# per the project convention -- never mix v1 and v2 results in one folder.
MODEL_FAMILY = "vit_only_v2"
SEEDS_TO_RUN = [42, 123, 7]

CLASS_NAMES = ["Diverticulosis", "Neoplasm", "Peritonitis", "Ureters"]
IMG_SIZE = 448
BATCH_SIZE = 16          # T4-safe for ViT-Small at 448x448; lower to 8 if you hit OOM
NUM_EPOCHS = 30          # ceiling -- early stopping + resume mean this is rarely fully used
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 6             # epochs without val_acc improvement before stopping early

# Points at the CORRECTED split -- do not change back to dataset_split.json (v1).
SPLIT_JSON_PATH    = os.path.join(EXPERIMENTS_ROOT, "dataset_split_v2.json")
MANIFEST_JSON_PATH = os.path.join(EXPERIMENTS_ROOT, "experiments_manifest.json")

assert os.path.exists(SPLIT_JSON_PATH), (
    "dataset_split_v2.json not found. Run NB0's duplicate-check/v2-split cells "
    "on this account first, or copy dataset_split_v2.json + checkpoint_utils.py "
    "in from the account that did."
)
assert os.path.exists(os.path.join(EXPERIMENTS_ROOT, "checkpoint_utils.py")), (
    "checkpoint_utils.py not found in EXPERIMENTS_ROOT. Same fix as above."
)

sys.path.insert(0, EXPERIMENTS_ROOT)
import checkpoint_utils as cku
print("checkpoint_utils imported OK from:", EXPERIMENTS_ROOT)


checkpoint_utils imported OK from: /content/drive/MyDrive/gastronet_experiments


### Local dataset copy
Reading thousands of individual image files directly off a mounted Google
Drive during training triggers Drive API rate-limiting partway through an
epoch (observed directly in NB1). Copying to local Colab disk once per
session avoids this. Copied per-class, per-file, with progress prints every
200 files -- a silent single `shutil.copytree()` call gives no feedback for
potentially 10+ minutes and makes it impossible to tell "slow" from "stuck."
This copy is session-temporary (wiped on disconnect) and must be redone each
fresh session -- that's expected, not a bug.


In [3]:
import shutil

LOCAL_DATASET_DIR = "/content/gastro_local_copy"

if not os.path.exists(LOCAL_DATASET_DIR):
    os.makedirs(LOCAL_DATASET_DIR)
    for cls in CLASS_NAMES:
        src_dir = os.path.join(RAW_DATASET_DIR, cls)
        dst_dir = os.path.join(LOCAL_DATASET_DIR, cls)
        os.makedirs(dst_dir, exist_ok=True)
        files = os.listdir(src_dir)
        print(f"Copying class '{cls}': {len(files)} files")
        for i, fname in enumerate(files):
            shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
            if (i + 1) % 200 == 0:
                print(f"  [{cls}] copied {i+1}/{len(files)}")
    print("Local copy complete.")
else:
    print("Local copy already exists this session, skipping copy.")


Copying class 'Diverticulosis': 1000 files
  [Diverticulosis] copied 200/1000
  [Diverticulosis] copied 400/1000
  [Diverticulosis] copied 600/1000
  [Diverticulosis] copied 800/1000
  [Diverticulosis] copied 1000/1000
Copying class 'Neoplasm': 1000 files
  [Neoplasm] copied 200/1000
  [Neoplasm] copied 400/1000
  [Neoplasm] copied 600/1000
  [Neoplasm] copied 800/1000
  [Neoplasm] copied 1000/1000
Copying class 'Peritonitis': 1000 files
  [Peritonitis] copied 200/1000
  [Peritonitis] copied 400/1000
  [Peritonitis] copied 600/1000
  [Peritonitis] copied 800/1000
  [Peritonitis] copied 1000/1000
Copying class 'Ureters': 1000 files
  [Ureters] copied 200/1000
  [Ureters] copied 400/1000
  [Ureters] copied 600/1000
  [Ureters] copied 800/1000
  [Ureters] copied 1000/1000
Local copy complete.


In [4]:
with open(SPLIT_JSON_PATH) as f:
    split_payload = json.load(f)

SPLIT_HASH = split_payload["split_hash"]
split = split_payload["split"]

assert split_payload["class_names"] == CLASS_NAMES, "Class name mismatch with locked split!"

print("Loaded split_hash (v2):", SPLIT_HASH)
if "derived_from" in split_payload:
    print("Derived from v1 split_hash:", split_payload["derived_from"])
    print("Fix note:", split_payload.get("fix_note", ""))
for k in ["train", "val", "test"]:
    print(f"  {k}: {len(split[k])} images")

# Remap paths to the local copy for actual file reads during training.
# dataset_split_v2.json itself is untouched -- it still stores canonical
# Drive paths, and SPLIT_HASH above was computed from those.
def remap_to_local(entries):
    return [(p.replace(RAW_DATASET_DIR, LOCAL_DATASET_DIR), cls) for p, cls in entries]

split = {k: remap_to_local(v) for k, v in split.items()}
print("Paths remapped to local copy, e.g.:", split["train"][0][0])


Loaded split_hash (v2): d6e80caa29bff18856ae93ea635b4650
Derived from v1 split_hash: 5ba9dc1cc68a1202cd0bb0446e457ab0
Fix note: Removed 2 duplicate file(s) that appeared in multiple splits in v1; duplicates kept in test, removed from train/val.
  train: 3198 images
  val: 400 images
  test: 400 images
Paths remapped to local copy, e.g.: /content/gastro_local_copy/Diverticulosis/08648b7f-37b9-406e-b1c3-03683c3222af.jpg


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

class GastroDataset(Dataset):
    def __init__(self, entries, transform):
        self.entries = entries
        self.transform = transform

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        path, cls = self.entries[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        label = class_to_idx[cls]
        return img, label

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Loaders are built in the cell below, AFTER the ViT input-size decision --
# torchvision's vit_b_16 pretrained weights expect 224x224, not the 448x448
# used elsewhere in this project for the CNN, so we deliberately don't build
# loaders here yet. See the flagged markdown cell right after model-building.


Device: cuda


In [7]:
# DECISION LOCKED: ViT-Small (not ViT-Base), installed via timm, since this
# needs to size-match the ViT branch that will live inside hybrid_crossattn
# later. Also locked: Option C for the image-size question -- each branch
# keeps its own native input resolution (CNN=448, ViT=224) rather than
# forcing both to match. This notebook already trains at 224 natively for
# ViT, so no further change is needed here for that decision -- it only
# affects how hybrid_crossattn's data pipeline is built later (two resized
# copies of each image, one per branch).

!pip install timm --break-system-packages -q

import timm
import torch.nn as nn

def build_vit_model(num_classes=len(CLASS_NAMES)):
    model = timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=num_classes)
    return model

IMG_SIZE_VIT = 224   # ViT-Small's native pretrained input size, kept as this
                      # notebook's own local variable -- does not affect
                      # IMG_SIZE (448) used elsewhere for the CNN.

print("Model: ViT-Small (timm, vit_small_patch16_224), pretrained, "
      f"{sum(p.numel() for p in build_vit_model().parameters()):,} params")


Model: ViT-Small (timm, vit_small_patch16_224), pretrained, 21,667,204 params


### Decisions locked for this notebook (confirmed)

- **ViT-Small**, not ViT-Base, via `timm` -- size-matches the branch that
  will be used inside `hybrid_crossattn`.
- **Option C** for image size: this ViT branch trains natively at 224x224
  (its pretrained size); the CNN (NB1) trains natively at 448x448. Neither
  is forced to match the other in these standalone baseline notebooks.
  `hybrid_crossattn` (NB4/NB5) will feed each branch its own natively-sized
  copy of the same image internally -- no resolution compromise on either
  side, and no need to re-run NB1 or this notebook again once fusion work
  starts.


In [8]:
# Built at IMG_SIZE_VIT (224), not the project-wide IMG_SIZE (448) used for
# the CNN, per the flag above -- this notebook's own choice, isolated to
# this notebook's variables only.

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_VIT, IMG_SIZE_VIT)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_VIT, IMG_SIZE_VIT)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = GastroDataset(split["train"], train_transform)
val_ds   = GastroDataset(split["val"], eval_transform)
test_ds  = GastroDataset(split["test"], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Built at {IMG_SIZE_VIT}x{IMG_SIZE_VIT}. "
      f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} batches/epoch={len(train_loader)}")


Built at 224x224. train=3198 val=400 test=400 batches/epoch=200


In [9]:
import time

def run_epoch(model, optimizer, scaler, criterion, loader, train_mode, print_every=20):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    start_time = time.time()
    last_print_time = start_time

    with torch.set_grad_enabled(train_mode):
        for batch_idx, (imgs, labels) in enumerate(loader):
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            if train_mode:
                optimizer.zero_grad()

            with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)

            if (batch_idx + 1) % print_every == 0:
                now = time.time()
                chunk_time = now - last_print_time
                last_print_time = now
                running_acc = correct / total
                mode_str = "train" if train_mode else "val"
                print(f"    [{mode_str}] batch {batch_idx+1}/{len(loader)} "
                      f"| running_acc={running_acc:.4f} "
                      f"| this_chunk={chunk_time:.1f}s | total_elapsed={now - start_time:.1f}s")

    return total_loss / total, correct / total


In [11]:
all_seed_results = {}

for SEED in SEEDS_TO_RUN:
    seed_exp_dir = cku.get_experiment_dir(EXPERIMENTS_ROOT, MODEL_FAMILY, SEED)

    if os.path.exists(os.path.join(seed_exp_dir, "results.json")):
        print(f"Seed {SEED} already has results.json, skipping.")
        with open(os.path.join(seed_exp_dir, "results.json")) as f:
            all_seed_results[SEED] = json.load(f)
        continue

    print(f"\n{'='*60}\nStarting SEED={SEED}\n{'='*60}")

    torch.manual_seed(SEED)
    model = build_vit_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    criterion = nn.CrossEntropyLoss()

    start_epoch, best_val_acc, history = cku.resume_or_start(
        seed_exp_dir, model, optimizer, scheduler, scaler, map_location=device
    )
    cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                       status="resumed" if start_epoch > 0 else "started",
                       best_val_acc=best_val_acc, drive_path=seed_exp_dir)

    epochs_without_improvement = 0

    try:
        for epoch in range(start_epoch, NUM_EPOCHS):
            train_loss, train_acc = run_epoch(model, optimizer, scaler, criterion, train_loader, train_mode=True)
            val_loss, val_acc = run_epoch(model, optimizer, scaler, criterion, val_loader, train_mode=False)
            scheduler.step()

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            print(f"[seed {SEED}] Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

            cku.save_latest(seed_exp_dir, epoch, model, optimizer, scheduler, scaler, best_val_acc, history)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                epochs_without_improvement = 0
                config = {
                    "model_family": MODEL_FAMILY, "seed": SEED, "split_hash": SPLIT_HASH,
                    "account_tag": ACCOUNT_TAG, "notebook_name": NOTEBOOK_NAME,
                    "img_size": IMG_SIZE_VIT, "batch_size": BATCH_SIZE,
                    "learning_rate": LEARNING_RATE, "class_names": CLASS_NAMES,
                }
                cku.save_best(seed_exp_dir, model, epoch, best_val_acc, config)
                cku.save_config(seed_exp_dir, config)
                print(f"  -> new best_val_acc={best_val_acc:.4f}, saved best.pt")
            else:
                epochs_without_improvement += 1
                print(f"  -> no improvement for {epochs_without_improvement}/{PATIENCE} epochs")
                if epochs_without_improvement >= PATIENCE:
                    print(f"Stopping early for seed {SEED}: no improvement for {PATIENCE} epochs.")
                    cku.save_history(seed_exp_dir, history)
                    break

            cku.save_history(seed_exp_dir, history)

    except KeyboardInterrupt:
        cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                           status="interrupted", best_val_acc=best_val_acc, drive_path=seed_exp_dir,
                           note="Manually interrupted -- latest.pt has current state, safe to resume.")
        print(f"Interrupted during seed {SEED}. Re-run this cell to resume from the last completed epoch.")
        break  # stop the whole multi-seed loop, don't silently move to the next seed

    # Test-set evaluation for this seed
    best_ckpt = cku.load_best(seed_exp_dir, map_location=device)
    cku.assert_split_hash_matches(best_ckpt["config"], SPLIT_HASH)
    eval_model = build_vit_model().to(device)
    eval_model.load_state_dict(best_ckpt["model_state_dict"])
    eval_model.eval()

    correct, total, preds_all, labels_all = 0, 0, [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = eval_model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
            preds_all.extend(preds.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())

    test_acc = correct / total
    results = {
        "model_family": MODEL_FAMILY, "seed": SEED, "split_hash": SPLIT_HASH,
        "account_tag": ACCOUNT_TAG, "best_epoch": best_ckpt["epoch"],
        "best_val_acc": best_ckpt["best_val_acc"], "test_accuracy": test_acc,
        "predictions": preds_all, "labels": labels_all, "class_names": CLASS_NAMES,
    }
    cku.save_results(seed_exp_dir, results)
    all_seed_results[SEED] = results

    cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                       status="completed", best_val_acc=best_val_acc, drive_path=seed_exp_dir)

    # Free this seed's latest.pt immediately -- at most one large checkpoint
    # alive on Drive at any point during the multi-seed loop.
    cku.finalize_experiment(seed_exp_dir)
    print(f"Seed {SEED} done. test_acc={test_acc:.4f}. latest.pt cleaned up.\n")

print("\nSeeds completed this run:")
for s, r in all_seed_results.items():
    print(f"  seed {s}: test_accuracy={r['test_accuracy']:.4f}, best_val_acc={r['best_val_acc']:.4f}")


Seed 42 already has results.json, skipping.
Seed 123 already has results.json, skipping.
Seed 7 already has results.json, skipping.

Seeds completed this run:
  seed 42: test_accuracy=0.9625, best_val_acc=0.9700
  seed 123: test_accuracy=0.9725, best_val_acc=0.9625
  seed 7: test_accuracy=0.9800, best_val_acc=0.9675


In [12]:
cku.manifest_summary(MANIFEST_JSON_PATH)


_setup             seed_0      -> completed    acc=None account=acct_A notebook=NB0_setup_and_split_lock at 2026-09-07 10:15:46
cnn_only           seed_7      -> completed    acc=0.985 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:56:38
cnn_only           seed_42     -> resumed      acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:26:57
cnn_only           seed_123    -> completed    acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:42:24
cnn_only_v2        seed_7      -> completed    acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-07 11:45:05
cnn_only_v2        seed_42     -> completed    acc=0.985 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-07 11:01:48
cnn_only_v2        seed_123    -> completed    acc=0.98 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-07 11:18:20
vit_only_v2        seed_7      -> completed    acc=0.9675 account=acct_A notebook=NB2_vit_only_baseline at 2026-09

In [13]:
import numpy as np
import glob

all_results = []
for rf in sorted(glob.glob(os.path.join(EXPERIMENTS_ROOT, MODEL_FAMILY, "seed_*", "results.json"))):
    with open(rf) as f:
        all_results.append(json.load(f))

test_accs = [r["test_accuracy"] for r in all_results]
seeds_found = [r["seed"] for r in all_results]

print(f"{MODEL_FAMILY} -- seeds found: {seeds_found}")
print(f"Test accuracies: {[round(a, 4) for a in test_accs]}")
if len(test_accs) > 1:
    print(f"Mean: {np.mean(test_accs):.4f}  Std: {np.std(test_accs, ddof=1):.4f}")
else:
    print("Only one seed found so far.")


vit_only_v2 -- seeds found: [123, 42, 7]
Test accuracies: [0.9725, 0.9625, 0.98]
Mean: 0.9717  Std: 0.0088


## After this notebook finishes

1. **Duplicate-file check is not re-run here** -- it was already handled at
   the split level in NB0 (v2 split has the leak fixed). No per-notebook
   action needed.
2. **Check `results.json` for each seed** in `gastronet_experiments/vit_only_v2/seed_{42,123,7}/`
   -- confirm `test_accuracy`/`best_val_acc` are sane, and watch the
   train/val gap in `history.json` for the same overfitting pattern seen in
   NB1 (train_acc pinned near 100% while val_acc plateaus). Early stopping
   should catch this automatically now, but worth a glance.
3. **Mean ± std**: the last code cell prints this directly. Compare against
   `cnn_only_v2`'s mean±std once that's re-run (see project handoff doc,
   Section 10, step 1) -- these two ablation baselines are the first real
   apples-to-apples comparison on the clean split.
4. **If Colab disconnects mid-run**: just re-run the multi-seed cell. It
   skips any seed with an existing `results.json` and resumes the
   in-progress seed from its `latest.pt` via `resume_or_start`.
5. **Next: NB3 (`hybrid_concat_v2`)** -- CNN (EfficientNet-B4, 448) + ViT-Small (224) features concatenated, then classified. Same multi-seed pattern as this notebook. Per the locked Option C decision, NB3's dataset pipeline will feed each branch its own natively-sized copy of the same image (two resize transforms per sample instead of one) -- not a resolution compromise on either side.
